In [7]:
import contextlib
import io
import logging
from io import BytesIO
from pathlib import Path

import h5py
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
from IPython.display import Image as IPyImage
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from mlpng.generator import Generator

In [8]:
# Main controls
SIM_INDEX = 0
PHI_SCALE = 1
SHOW_DIFF = True  # Show diff plots: left=difference from baseline, right=current

# Select any subset of: local, equilateral, orthogonal
SHAPES = ["local", "equilateral", "orthogonal"]

# Per-shape f_nl values used in map construction
FNL_BY_SHAPE = {
    "local": 100.0,
    "equilateral": 100.0,
    "orthogonal": 100.0,
}

NSIDE_CONFIGS = {
    # 64: ("settings/n64.json", "data/data/l191_n64_T_10000_p1.0.hdf5"),
    256: ("settings/n256.json", "data/data/l767_n256_T_10000_p1.0.hdf5"),
}

In [9]:
# planck colormapgs from https://raw.githubusercontent.com/zonca/paperplots/master/data/Planck_Parchment_RGB.txt
# https://www.zonca.dev/posts/2013-09-10-planck-cmb-map-at-high-resolution

cmap = ListedColormap(np.loadtxt("data/Planck_Parchment_RGB.txt") / 255.0)
cmap.set_bad("gray")  # color of missing pixels
cmap.set_under("white")  # color of background

In [10]:
# Reduce lensing library chatter in notebook output.
for _logger_name in [
    "lenspyx",
    "lenspyx.remapping",
    "lenspyx.remapping.utils_geom",
    "lenspyx.remapping.deflection",
]:
    logging.getLogger(_logger_name).setLevel(logging.ERROR)

ALLOWED_SHAPES = {"local", "equilateral", "orthogonal"}


def validate_config(shapes, fnl_by_shape):
    invalid = [shape for shape in shapes if shape not in ALLOWED_SHAPES]
    if invalid:
        raise ValueError(
            f"Invalid shapes: {invalid}. Allowed: {sorted(ALLOWED_SHAPES)}"
        )

    missing = [shape for shape in shapes if shape not in fnl_by_shape]
    if missing:
        raise ValueError(f"Missing f_nl values for shapes: {missing}")


def get_latex_label(param_name):
    """Convert parameter name to LaTeX format for display."""
    latex_map = {
        "fnl": r"$f_\mathrm{NL}$",
        "phi": r"$A_\mathrm{lens}$",
    }
    return latex_map.get(param_name.lower(), param_name)


def load_single_sim_data(data_file, sim_index, shape):
    with h5py.File(data_file, "r") as f:
        alm_l = np.asarray(f[f"alm_l/unlensed/{shape}"][sim_index], dtype=np.complex128)
        alm_nl = np.asarray(
            f[f"alm_nl/unlensed/{shape}"][sim_index], dtype=np.complex128
        )
        alm_phi = np.asarray(f["alm_phi"][sim_index], dtype=np.complex128)

    return alm_l, alm_nl, alm_phi


def lens_alm_to_map(gen, alm, alm_phi, phi_scale):
    np.random.seed(42)

    alm_arr = np.asarray(alm, dtype=np.complex128)
    if alm_arr.ndim == 1:
        alm_arr = alm_arr[np.newaxis, np.newaxis, :]
    elif alm_arr.ndim == 2:
        alm_arr = alm_arr[np.newaxis, :, :]

    if phi_scale == 0:
        return np.array(hp.alm2map(alm_arr[0, 0], gen.nside, pol=False))

    phi_arr = np.asarray(alm_phi, dtype=np.complex128)
    if phi_arr.ndim == 1:
        phi_arr = phi_arr[np.newaxis, :]

    nsims_input = alm_arr.shape[0]
    if phi_arr.shape[0] != nsims_input:
        if phi_arr.shape[0] == 1:
            phi_arr = np.repeat(phi_arr, nsims_input, axis=0)
        else:
            raise ValueError(
                f"alm_phi first dimension ({phi_arr.shape[0]}) must match alm first dimension ({nsims_input})"
            )

    phi_arr = phi_arr * float(phi_scale)

    nsims_original = gen.nsims
    try:
        gen.nsims = nsims_input
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(
            io.StringIO()
        ):
            lensed_alm, _ = gen.lens_alms(alm_arr, phi_arr)
    finally:
        gen.nsims = nsims_original

    return np.array(hp.alm2map(lensed_alm[0, 0], gen.nside, pol=False))


def render_mollview_frame(t_map, title, cmap="coolwarm", vmin=None, vmax=None, dpi=120):
    """Render a mollview frame as a PIL Image with white background."""
    fig = plt.figure(figsize=(6, 4), facecolor="white")
    hp.mollview(
        t_map,
        title=title,
        cmap=cmap,
        fig=fig.number,
        bgcolor="white",
        notext=False,
        remove_dip=True,
        remove_mono=True,
        min=vmin,
        max=vmax,
    )
    plt.gcf().patch.set_facecolor("white")

    buf = BytesIO()
    plt.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.close(fig)
    buf.seek(0)

    return Image.open(buf).convert("RGB")


def render_mollview_diff_split(
    diff_map,
    current_map,
    title,
    cmap="coolwarm",
    vmin=None,
    vmax=None,
    dpi=120,
):
    """Render a mollview diff-split frame as a PIL Image with white background."""
    npix = len(diff_map)
    nside = hp.npix2nside(npix)

    # Get the longitude (phi) of each pixel
    theta, phi = hp.pix2ang(nside, np.arange(npix))

    # Create masks: left half (diff, phi < pi) and right half (current, phi >= pi)
    left_mask = phi < np.pi
    right_mask = phi >= np.pi

    # Create masked arrays
    diff_masked = hp.ma(diff_map)
    diff_masked.mask = np.logical_not(left_mask)
    current_masked = hp.ma(current_map)
    current_masked.mask = np.logical_not(right_mask)

    combined_map = np.where(left_mask, diff_map, current_map)

    # Create figure with single mollweide projection
    fig = plt.figure(figsize=(6, 4), facecolor="white")

    # Plot difference map on left half
    old_titlesize = plt.rcParams["axes.titlesize"]
    plt.rcParams["axes.titlesize"] = 24
    hp.mollview(
        combined_map,
        title=title,
        cmap=cmap,
        fig=fig.number,
        bgcolor="white",
        min=vmin,
        max=vmax,
        remove_dip=True,
        remove_mono=True,
        cbar=False,
        notext=True,
    )

    buf = BytesIO()
    plt.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none",
    )
    plt.close(fig)
    buf.seek(0)

    plt.rcParams["axes.titlesize"] = old_titlesize
    return Image.open(buf).convert("RGB")


def render_latex_text(text, fontsize=28, dpi=100):
    """
    Render LaTeX text using matplotlib and return as PIL Image with minimal size.

    Args:
        text: LaTeX string (e.g., r"$f_\mathrm{NL}$=100")
        fontsize: Font size for the text
        dpi: DPI for rendering

    Returns:
        PIL Image with rendered text on transparent background, cropped to minimal size
    """
    # Create a minimal figure with just text
    fig = plt.figure(figsize=(4, 1), dpi=dpi, facecolor="none")
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")

    # Render text with LaTeX support
    ax.text(
        0.5,
        0.5,
        text,
        fontsize=fontsize,
        ha="center",
        va="center",
        transform=ax.transAxes,
        color="black",
    )

    # Convert to PIL Image
    buf = BytesIO()
    plt.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0,
        facecolor="none",
        edgecolor="none",
        transparent=True,
    )
    plt.close(fig)
    buf.seek(0)

    img = Image.open(buf).convert("RGBA")

    # Crop to minimal bounding box of non-transparent pixels
    bbox = img.getbbox()
    if bbox:
        img = img.crop(bbox)

    return img


def composite_image_on_grid(grid_img, overlay_img, x, y):
    """
    Composite an overlay image onto a grid image at position (x, y).

    Args:
        grid_img: PIL Image (the background grid)
        overlay_img: PIL Image with RGBA channels (the overlay)
        x, y: Position to place the overlay (center)

    Returns:
        Modified grid_img with overlay composited
    """
    # Convert grid to RGBA if needed
    if grid_img.mode != "RGBA":
        grid_img = grid_img.convert("RGBA")

    # Resize overlay to fit
    overlay_img = overlay_img.resize((overlay_img.width, overlay_img.height))

    # Calculate position (center the overlay at x, y)
    x_pos = x - overlay_img.width // 2
    y_pos = y - overlay_img.height // 2

    # Composite using alpha blending
    grid_img.paste(overlay_img, (x_pos, y_pos), overlay_img)

    return grid_img


def create_grid_frame(
    frame_tl,
    frame_tr,
    frame_bl,
    frame_br,
    param_label,
    dpi=80,
    add_text_overlay=False,
    show_diff=False,
):
    """
    Create a 2x2 grid image from 4 frames with optional centered parameter text overlay.

    Args:
        frame_tl, frame_tr, frame_bl, frame_br: PIL Image frames (top-left, top-right, bottom-left, bottom-right)
        param_label: Text label for parameter value to overlay (e.g., "$f_{NL}$=100")
        dpi: DPI for output
        add_text_overlay: If True, add centered parameter text overlay to grid (rendered with LaTeX)
        show_diff: If True, add diff-side labels to bottom two plots (rendered with LaTeX)
    """
    # Convert frames to numpy arrays for easy manipulation
    frame_arrays = [
        np.array(f.convert("RGB")) for f in [frame_tl, frame_tr, frame_bl, frame_br]
    ]

    fig, axes = plt.subplots(2, 2, figsize=(8, 6), facecolor="white")

    # Titles for each subplot (shape names, but displayed with empty titles to match individual frames)
    titles = ["local", "equilateral", "orthogonal", "combined"]
    frames_layout = [frame_arrays[0], frame_arrays[1], frame_arrays[2], frame_arrays[3]]

    for ax, img, title in zip(axes.flat, frames_layout, titles):
        ax.imshow(img)
        ax.set_title("", fontsize=12, fontweight="bold")
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 1])

    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)

    # Load the grid image
    grid_img = Image.open(buf).convert("RGB")
    img_width, img_height = grid_img.size

    # Add main parameter label using LaTeX rendering (centered vertically and horizontally)
    if add_text_overlay:
        param_overlay = render_latex_text(param_label, fontsize=28, dpi=dpi)
        # Center the overlay in the grid
        grid_img = grid_img.convert("RGBA")
        center_x = img_width // 2
        center_y = img_height // 2
        x_pos = center_x - param_overlay.width // 2
        y_pos = center_y - param_overlay.height // 2
        grid_img.paste(param_overlay, (x_pos, y_pos), param_overlay)
        grid_img = grid_img.convert("RGB")

    # Add diff-side labels to bottom two plots using LaTeX rendering if show_diff is True
    if show_diff:
        grid_img = grid_img.convert("RGBA")
        w = img_width // 8

        left_label_text = r"$T^{\mathrm{NG}}(\hat{n}) - T^{\mathrm{G}}(\hat{n})$"
        right_label_text = r"$T^{\mathrm{NG}}(\hat{n})$"
        left_overlay = render_latex_text(left_label_text, fontsize=12, dpi=dpi)
        right_overlay = render_latex_text(right_label_text, fontsize=12, dpi=dpi)

        label_y = img_height - left_overlay.height
        left_x = w - left_overlay.width // 2 + 20
        right_x = 3 * w - right_overlay.width // 2 - 20
        combined_left_x = 4 * w + left_x
        combined_right_x = 4 * w + right_x

        grid_img.paste(left_overlay, (max(0, left_x), label_y), left_overlay)
        grid_img.paste(right_overlay, (max(0, right_x), label_y), right_overlay)
        grid_img.paste(left_overlay, (max(0, combined_left_x), label_y), left_overlay)
        grid_img.paste(
            right_overlay, (min(img_width, combined_right_x), label_y), right_overlay
        )
        grid_img = grid_img.convert("RGB")

    return grid_img


def save_gif(frames, out_file, duration, loop=0):
    """Save frames as GIF (white background, no transparency)."""
    if not frames:
        raise ValueError("No frames were generated")

    rgb_frames = [frame.convert("RGB") for frame in frames]
    rgb_frames[0].save(
        out_file,
        save_all=True,
        append_images=rgb_frames[1:],
        duration=duration,
        loop=loop,
        optimize=True,
    )


def generate_animation(
    gen,
    alm_base,
    alm_nl_by_shape,
    alm_phi,
    param_values,
    param_name,
    shapes,
    fnl_by_shape,
    output_dir,
    anim_settings,
    is_fnl_anim=True,
    show_diff=False,
):
    """
    Generate and save animations for per-shape and combined configurations.
    Also creates 2x2 grid GIFs combining all 4 animations with simplified labels.

    Args:
        show_diff: If True, create split-view plots with baseline diff on left and current on right.
        is_fnl_anim: If True, generate per-shape and combined animations (f_nl mode).
                     If False, generate only combined animation (phi mode).
    """
    param_name_lower = param_name.lower()
    latex_label = get_latex_label(param_name)

    # Calculate frame duration including initial and final delays
    initial_delay = anim_settings.get("INITIAL_DELAY", 2)
    final_delay = anim_settings.get("FINAL_DELAY", 0)
    total_delay_sec = initial_delay + final_delay
    anim_sec = anim_settings["SECONDS"]

    # Ensure animation time is at least as long as total delays
    anim_time_available = max(
        anim_sec + total_delay_sec, 0.1
    )  # Minimum 0.1 sec for frames

    frame_duration_ms = int(round((anim_time_available * 1000) / len(param_values)))

    # Clamp to PIL GIF format limits (0-65535 milliseconds)
    frame_duration_ms = max(10, min(frame_duration_ms, 65535))
    nside = gen.nside

    # For fNL animations: generate per-shape and combined
    # For Phi animations: only generate combined (no per-shape)
    if is_fnl_anim:
        # Store all frames for each shape to build grid later
        all_shape_frames = {shape: [] for shape in shapes}
        combined_frames_list = []

        for shape in shapes:
            frames = []
            preview_maps = []
            baseline_map = np.array(
                hp.alm2map(alm_base[0], gen.nside, pol=False)
            )  # Will store the first map for diff calculation

            phi_scale_val = anim_settings.get("PHI_SCALE", 0)
            for idx, param_val in enumerate(param_values):
                alm_frame = alm_base + float(param_val) * alm_nl_by_shape[shape]
                t_map = lens_alm_to_map(gen, alm_frame, alm_phi, phi_scale_val)
                preview_maps.append(np.asarray(t_map))

            stacked = np.stack(preview_maps, axis=0)
            vmin = float(np.percentile(stacked, 1.0))
            vmax = float(np.percentile(stacked, 99.0))

            for param_val, t_map in zip(param_values, preview_maps):
                # Construct title with shape and parameter value
                title = f"{shape}"

                if show_diff:
                    diff_map = t_map - baseline_map
                    frame = render_mollview_diff_split(
                        diff_map,
                        t_map,
                        title,
                        cmap=anim_settings["CMAP"],
                        vmin=vmin,
                        vmax=vmax,
                        dpi=anim_settings["DPI"],
                    )
                else:
                    frame = render_mollview_frame(
                        t_map,
                        title,
                        cmap=anim_settings["CMAP"],
                        vmin=vmin,
                        vmax=vmax,
                        dpi=anim_settings["DPI"],
                    )
                frames.append(frame)
                all_shape_frames[shape].append(frame)

            # Add initial frame duplicates for delay
            if initial_delay > 0:
                first_frame = frames[0]
                n_repeats = int(initial_delay * 1000 / frame_duration_ms)
                for _ in range(n_repeats):
                    frames.insert(0, first_frame)

            # Add final frame duplicates for freeze effect
            if anim_settings["FINAL_DELAY"] > 0:
                final_frame = frames[-1]
                n_repeats = int(anim_settings["FINAL_DELAY"] * 1000 / frame_duration_ms)
                for _ in range(n_repeats):
                    frames.append(final_frame)

            out_suffix = "_diff" if show_diff else ""
            out_file = (
                output_dir / f"{param_name_lower}_{shape}{out_suffix}_n{nside}.gif"
            )
            save_gif(
                frames,
                out_file,
                duration=frame_duration_ms,
                loop=anim_settings["LOOP"],
            )

        # Combined-shape animation
        frames_combined = []
        preview_maps_combined = []

        for idx, param_val in enumerate(param_values):
            combined_nl = np.zeros_like(alm_base)
            for shape in shapes:
                combined_nl += alm_nl_by_shape[shape]
            alm_combined = alm_base + float(param_val) * combined_nl
            t_map = lens_alm_to_map(
                gen, alm_combined, alm_phi, anim_settings.get("PHI_SCALE", 0)
            )
            preview_maps_combined.append(np.asarray(t_map))

        combined_stacked = np.stack(preview_maps_combined, axis=0)
        combined_vmin = float(np.percentile(combined_stacked, 1.0))
        combined_vmax = float(np.percentile(combined_stacked, 99.0))

        for param_val, t_map in zip(param_values, preview_maps_combined):
            title = f"combined"

            if show_diff:
                diff_map = t_map - baseline_map
                frame = render_mollview_diff_split(
                    diff_map,
                    t_map,
                    title,
                    cmap=anim_settings["CMAP"],
                    vmin=combined_vmin,
                    vmax=combined_vmax,
                    dpi=anim_settings["DPI"],
                )
            else:
                frame = render_mollview_frame(
                    t_map,
                    title,
                    cmap=anim_settings["CMAP"],
                    vmin=combined_vmin,
                    vmax=combined_vmax,
                    dpi=anim_settings["DPI"],
                )
            frames_combined.append(frame)
            combined_frames_list.append(frame)

        # Add initial frame duplicates for delay
        if initial_delay > 0:
            first_frame = frames_combined[0]
            n_repeats = int(initial_delay * 1000 / frame_duration_ms)
            for _ in range(n_repeats):
                frames_combined.insert(0, first_frame)

        # Add final frame duplicates for freeze effect
        if anim_settings["FINAL_DELAY"] > 0:
            final_frame = frames_combined[-1]
            n_repeats = int(anim_settings["FINAL_DELAY"] * 1000 / frame_duration_ms)
            for _ in range(n_repeats):
                frames_combined.append(final_frame)

        out_suffix = "_diff" if show_diff else ""
        combined_out_file = (
            output_dir / f"{param_name_lower}_combined{out_suffix}_n{nside}.gif"
        )
        save_gif(
            frames_combined,
            combined_out_file,
            duration=frame_duration_ms,
            loop=anim_settings["LOOP"],
        )

        # Create grid GIF combining all 4 animations
        grid_frames = []

        for idx, param_val in enumerate(param_values):
            # Get frames at this index for each shape + combined
            # Account for initial delay frames that were prepended
            n_initial = (
                int(initial_delay * 1000 / frame_duration_ms)
                if initial_delay > 0
                else 0
            )
            frame_idx = n_initial + idx

            frame_shape0 = all_shape_frames[shapes[0]][idx]
            frame_shape1 = all_shape_frames[shapes[1]][idx]
            frame_shape2 = all_shape_frames[shapes[2]][idx]
            frame_combined = combined_frames_list[idx]

            param_label = f"{latex_label}={int(param_val)}"
            grid_frame = create_grid_frame(
                frame_shape0,
                frame_shape1,
                frame_shape2,
                frame_combined,
                param_label,
                dpi=anim_settings["DPI"],
                add_text_overlay=True,
                show_diff=show_diff,
            )
            grid_frames.append(grid_frame)

        # Add initial frame duplicates for grid GIF
        if initial_delay > 0:
            first_grid = grid_frames[0]
            n_repeats = int(initial_delay * 1000 / frame_duration_ms)
            for _ in range(n_repeats):
                grid_frames.insert(0, first_grid)

        # Add final frame duplicates for grid GIF too
        if anim_settings["FINAL_DELAY"] > 0:
            final_grid = grid_frames[-1]
            n_repeats = int(anim_settings["FINAL_DELAY"] * 1000 / frame_duration_ms)
            for _ in range(n_repeats):
                grid_frames.append(final_grid)

        grid_out_file = output_dir / f"{param_name_lower}_grid_n{nside}.gif"
        save_gif(
            grid_frames,
            grid_out_file,
            duration=frame_duration_ms,
            loop=anim_settings["LOOP"],
        )

    else:
        # Phi-scale animation: generate only combined map
        frames_combined = []
        preview_maps_combined = []

        combined_alm = np.array(alm_base, copy=True)
        for shape in shapes:
            combined_alm += float(fnl_by_shape[shape]) * alm_nl_by_shape[shape]

        for idx, param_val in enumerate(param_values):
            t_map = lens_alm_to_map(gen, combined_alm, alm_phi, float(param_val))
            preview_maps_combined.append(np.asarray(t_map))

        combined_stacked = np.stack(preview_maps_combined, axis=0)
        combined_vmin = float(np.percentile(combined_stacked, 1.0))
        combined_vmax = float(np.percentile(combined_stacked, 99.0))

        for param_val, t_map in zip(param_values, preview_maps_combined):
            # Title always shows phi_scale value for combined-only phi animations
            title = f"Effects of Lensing ({latex_label} = {int(param_val)})"

            frame = render_mollview_frame(
                t_map,
                title,
                cmap=anim_settings["CMAP"],
                vmin=combined_vmin,
                vmax=combined_vmax,
                dpi=anim_settings["DPI"],
            )
            frames_combined.append(frame)

        # Add initial frame duplicates for delay
        if initial_delay > 0:
            first_frame = frames_combined[0]
            n_repeats = int(initial_delay * 1000 / frame_duration_ms)
            for _ in range(n_repeats):
                frames_combined.insert(0, first_frame)

        # Add final frame duplicates for freeze effect
        if anim_settings["FINAL_DELAY"] > 0:
            final_frame = frames_combined[-1]
            n_repeats = int(anim_settings["FINAL_DELAY"] * 1000 / frame_duration_ms)
            for _ in range(n_repeats):
                frames_combined.append(final_frame)

        combined_out_file = output_dir / f"{param_name_lower}_combined_n{nside}.gif"
        save_gif(
            frames_combined,
            combined_out_file,
            duration=frame_duration_ms,
            loop=anim_settings["LOOP"],
        )

## Fnl Animations

In [11]:
# ANIM_FNL_MIN = -5000.0
# ANIM_FNL_MAX = 5000.0
# ANIM_SECONDS = 20
# ANIM_STEPS = ANIM_SECONDS * 5 + 1
# ANIM_LOOP = 0
# ANIM_PHI_SCALE = 0
# ANIM_CMAP = cmap
# ANIM_DPI = 120
# ANIM_INITIAL_DELAY = 0
# ANIM_FINAL_DELAY = 5
# ANIM_OUTPUT_DIR = Path("data/plots/fnl_animations")

# # Create settings dict for animation function
# FNL_ANIM_SETTINGS = {
#     "STEPS": ANIM_STEPS,
#     "SECONDS": ANIM_SECONDS,
#     "LOOP": ANIM_LOOP,
#     "INITIAL_DELAY": ANIM_INITIAL_DELAY,
#     "FINAL_DELAY": ANIM_FINAL_DELAY,
#     "PHI_SCALE": ANIM_PHI_SCALE,
#     "CMAP": ANIM_CMAP,
#     "DPI": ANIM_DPI,
# }

# # Generate f_nl sequence
# fnl_values = np.linspace(ANIM_FNL_MIN, ANIM_FNL_MAX, ANIM_STEPS)
# ANIM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # Generate animations
# validate_config(SHAPES, FNL_BY_SHAPE)

# for nside, (settings_file, data_file) in NSIDE_CONFIGS.items():
#     gen = Generator(
#         argv=[settings_file, "--nsims", "10000", "--lensing", "--shapes", "all"]
#     )

#     ref_shape = SHAPES[0]
#     alm_l_ref, _, alm_phi = load_single_sim_data(data_file, SIM_INDEX, ref_shape)
#     alm_l_ref = np.asarray(alm_l_ref, dtype=np.complex128)

#     alm_nl_by_shape = {}
#     for shape in SHAPES:
#         _, alm_nl_shape, _ = load_single_sim_data(data_file, SIM_INDEX, shape)
#         alm_nl_by_shape[shape] = np.asarray(alm_nl_shape, dtype=np.complex128)

#     generate_animation(
#         gen,
#         alm_l_ref,
#         alm_nl_by_shape,
#         alm_phi,
#         fnl_values,
#         "fnl",
#         SHAPES,
#         FNL_BY_SHAPE,
#         ANIM_OUTPUT_DIR,
#         FNL_ANIM_SETTINGS,
#         is_fnl_anim=True,
#         show_diff=SHOW_DIFF,
#     )

## Phi-Scale Animations

In [12]:
# Phi-Scale Animation Configuration
PHI_ANIM_MIN = 0.0
PHI_ANIM_MAX = 300.0
PHI_ANIM_SECONDS = 20
PHI_ANIM_STEPS = PHI_ANIM_SECONDS * 5
PHI_ANIM_LOOP = 0
PHI_ANIM_CMAP = cmap
PHI_ANIM_DPI = 120
PHI_INITIAL_DELAY = 2
PHI_ANIM_FINAL_DELAY = 5

# Create settings dict for animation function
PHI_ANIM_SETTINGS = {
    "STEPS": PHI_ANIM_STEPS,
    "SECONDS": PHI_ANIM_SECONDS,
    "LOOP": PHI_ANIM_LOOP,
    "INITIAL_DELAY": PHI_INITIAL_DELAY,
    "FINAL_DELAY": PHI_ANIM_FINAL_DELAY,
    "CMAP": PHI_ANIM_CMAP,
    "DPI": PHI_ANIM_DPI,
}

# Generate phi_scale sequence
phi_scales = np.linspace(PHI_ANIM_MIN, PHI_ANIM_MAX, PHI_ANIM_STEPS)

# Generate animations
validate_config(SHAPES, FNL_BY_SHAPE)

for nside, (settings_file, data_file) in NSIDE_CONFIGS.items():
    gen = Generator(
        argv=[settings_file, "--nsims", "10000", "--lensing", "--shapes", "all"]
    )

    ref_shape = SHAPES[0]
    alm_l_ref, _, alm_phi = load_single_sim_data(data_file, SIM_INDEX, ref_shape)
    alm_l_ref = np.asarray(alm_l_ref, dtype=np.complex128)

    alm_nl_by_shape = {}
    for shape in SHAPES:
        _, alm_nl_shape, _ = load_single_sim_data(data_file, SIM_INDEX, shape)
        alm_nl_by_shape[shape] = np.asarray(alm_nl_shape, dtype=np.complex128)

    generate_animation(
        gen,
        alm_l_ref,
        alm_nl_by_shape,
        alm_phi,
        phi_scales,
        "phi",
        SHAPES,
        FNL_BY_SHAPE,
        ANIM_OUTPUT_DIR,
        PHI_ANIM_SETTINGS,
        is_fnl_anim=False,
        show_diff=False,
    )